In [ ]:
import torch
import transformers
import peft
import bitsandbytes
import datasets
import accelerate
import pandas
import tokenizers
import huggingface_hub
from importlib.metadata import version, PackageNotFoundError

def get_version(package_name):
    try:
        module = __import__(package_name)
        return module.__version__
    except (AttributeError, ImportError):
        try:
            return version(package_name)
        except PackageNotFoundError:
            return "Not Installed"

libraries = {
    "torch": torch.__version__,
    "cuda_version": torch.version.cuda,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "bitsandbytes": get_version("bitsandbytes"),
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "tokenizers": tokenizers.__version__,
    "huggingface-hub": huggingface_hub.__version__,
    "pandas": pandas.__version__,
}

print("### Reporte para requirements.txt ###")
for lib, ver in libraries.items():
    print(f"{lib}=={ver}")


### Reporte para requirements.txt ###
torch==2.1.1+cu121
cuda_version==12.1
transformers==4.40.2
peft==0.10.0
bitsandbytes==0.41.3
datasets==2.18.0
accelerate==0.28.0
tokenizers==0.19.1
huggingface-hub==0.20.3
pandas==2.2.0


## Librerías y Parámetros



In [ ]:
import torch
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from lowresource_llm_evaluation.LanguageDatasets import LanguageDataset
import time

LANGUAGE = "asturiano"
MODELO = "Qwen/Qwen2.5-7B-Instruct" 

# Define output directory for QLoRA adapter
date = time.localtime(time.time())
output_dir = f"./qlora_{LANGUAGE}__{MODELO.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}"

print(f"Model name set to: {MODELO}")
print(f"Output directory set to: {output_dir}")

2026-04-12 15:56:49.971624: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-12 15:56:49.972568: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-12 15:56:50.109459: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-12 15:56:50.391168: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-12 15:56:51.978443: W tensorflow/compiler/tf2

Model name set to: Qwen/Qwen2.5-7B-Instruct
Output directory set to: ./qlora_asturiano__Qwen2.5-7B-Instruct_04-12_15-56-52


## Cuantización del Modelo

Cargar un modelo pre-entrenado de Hugging Face y configurarlo para la cuantización de 4 bits, preparándolo para QLoRA.


In [ ]:
# 1. Define the 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# 2. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODELO, trust_remote_code=True)

# 3. Load the pre-trained language model with quantization
model = AutoModelForCausalLM.from_pretrained(
    MODELO,
    quantization_config=bnb_config,
    trust_remote_code=True,
    tie_word_embeddings=False # Added to silence the warning about tied weights
)

model.config.use_cache = False  # importante con Trainer
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Model loaded with 4-bit quantization: {model.__class__.__name__}")
print(f"Model device: {model.device}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/3.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Tokenizer loaded: Qwen2TokenizerFast
Model loaded with 4-bit quantization: Qwen2ForCausalLM
Model device: cuda:0


## Configuración QLoRA

Definir los hiperparámetros de QLoRA (por ejemplo, lora_r, lora_alpha, lora_dropout) y crear el modelo PEFT usando get_peft_model para envolver el modelo base cuantificado.


In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

peft_model = get_peft_model(model, lora_config)

print("QLoRA configuration applied.")
peft_model.print_trainable_parameters()

QLoRA configuration applied.
trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.26434798934534914


## Entrenamiento



In [ ]:

train, test = (LanguageDataset(LANGUAGE, filter_language_thr=0.15)
               .read_opus(source="NLLB", version=1)
               .filter_by_language(1024, top_k=10)
               .split(test_size=0.05, tokenizer=tokenizer))
del a
print("Dataset real cargado y tokenizado correctamente.")
print(f"Ejemplo tokenizado: {train[0]}")

[INFO] Descargando FastText LID-176 a /usr/local/lib/python3.11/dist-packages/lowresource_llm_evaluation/models/lid.176.ftz ...
[INFO] Modelo FastText descargado correctamente.
Descargando tatoeba para asturiano:


Completado con éxito


Map:   0%|          | 0/151 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Dataset real cargado y tokenizado correctamente.
Ejemplo tokenizado: {'input_ids': [13782, 78, 19478, 775, 1709, 91565, 1709, 1013, 21249, 404, 13, 31169, 409, 775, 10918, 272, 45257, 4791, 13, 63678, 511, 281, 361, 2739, 14370, 16449, 7061, 1709, 1375, 140766, 13, 6485, 1531, 44312, 28095, 511, 5208, 1375, 436, 53908, 1195, 436, 13, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 

In [ ]:
from transformers import TrainingArguments, Trainer

# 4. Configure TrainingArguments
training_args = TrainingArguments(
    output_dir=output_dir, # Use the output_dir defined previously
    per_device_train_batch_size=2, # Small batch size for mock data
    num_train_epochs=3, # Small number of epochs for quick demonstration
    learning_rate=1e-4,
    logging_steps=100, # Log every step
    fp16=True,
    bf16=False,
    do_eval=True, # No evaluation for this simple mock training
    save_strategy="epoch",
    report_to='none', # Do not report to any service
    optim="paged_adamw_32bit",
    gradient_checkpointing=True,
    gradient_accumulation_steps=8,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1
)

# 5. Create an instance of Trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test
)

# 6. Start the training process
trainer.train()

print("QLoRA training completed.")

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:429: UserWarning: torch.utils.checkpoint: please pass in use_reentrant=True or use_reentrant=False explicitly. The default value of use_reentrant will be updated to be False in the future. To maintain current behavior, pass use_reentrant=True. It is recommended that you use use_reentrant=False. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss


In [ ]:
print(train[0])

{'input_ids': [6582, 24141, 963, 84, 409, 1974, 318, 1167, 84, 19478, 2004, 49822, 3914, 264, 1187, 18415, 2962, 685, 3852, 24741, 3163, 14315, 379, 17796, 843, 62015, 1187, 326, 3732, 685, 312, 15999, 264, 775, 9323, 12088, 409, 922, 6, 22371, 18415, 2962, 685, 274, 97179, 32338, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643, 151643

### Exportar


In [ ]:
peft_model.save_pretrained(output_dir)
print(f"QLoRA adapter saved to {output_dir}")

## Exportación del LLM

In [ ]:
from peft import PeftModel

# Define a new directory for the fully fine-tuned model
finetuned_model_output_dir = "./finetuned_qlora_model"

# Merge the QLoRA adapter layers into the base model
# The peft_model object already contains the base model wrapped with the adapter
merged_model = peft_model.merge_and_unload()

# Save the merged model
merged_model.save_pretrained(finetuned_model_output_dir)

# Save the tokenizer
tokenizer.save_pretrained(finetuned_model_output_dir)

print(f"Fine-tuned model (merged) saved to: {finetuned_model_output_dir}")
print(f"Tokenizer saved to: {finetuned_model_output_dir}")

## Add `loadQlora` Function



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

def loadQlora(base_model_hf_name, qlora_adapter_path):
    # 1. Define the 4-bit quantization configuration
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Load the base pre-trained language model with quantization
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_hf_name,
        quantization_config=bnb_config,
        device_map='auto',
        trust_remote_code=True,
        tie_word_embeddings=False # Added to silence the warning
    )

    # 3. Load the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_hf_name, trust_remote_code=True)
    # Set pad_token_id if it's None, typically to eos_token_id for causal LMs
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # 4. Load the QLoRA adapter onto the base model
    peft_model = PeftModel.from_pretrained(base_model, qlora_adapter_path)

    print(f"Base model '{base_model_hf_name}' loaded with 4-bit quantization.")
    print(f"Tokenizer loaded.")
    print(f"QLoRA adapter loaded from '{qlora_adapter_path}'.")

    return peft_model, tokenizer

print("loadQlora function defined.")

## Interacción con el Modelo Afinado


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model and tokenizer
# Ensure finetuned_model_output_dir is defined from previous steps

# It's important to set device_map to 'auto' to ensure the model is loaded efficiently
# especially if a GPU is available. If not, it will default to CPU.
finetuned_model = AutoModelForCausalLM.from_pretrained(finetuned_model_output_dir, device_map='auto', trust_remote_code=True)
finetuned_tokenizer = AutoTokenizer.from_pretrained(finetuned_model_output_dir, trust_remote_code=True)

print(f"Fine-tuned model loaded from: {finetuned_model_output_dir}")
print(f"Fine-tuned tokenizer loaded from: {finetuned_model_output_dir}")
print(f"Model device: {finetuned_model.device}")

In [ ]:
def generate_text(prompt, model, tokenizer, max_length=100):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)

    # Move inputs to the model's device (CPU in this case)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    # Generate text
    output_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_length=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True, # Enable sampling for more diverse outputs
        top_k=50, # Consider top 50 tokens for sampling
        top_p=0.95, # Nucleus sampling
        temperature=0.7, # Controls randomness
        pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the generated sequence
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=True)

    return generated_text


In [ ]:
print("\n--- Reloading and interacting with the fine-tuned model using loadQlora ---")

# Use the loadQlora function to load the PEFT model and tokenizer
# MODELO and output_dir are defined in previous cells
peft_finetuned_model, peft_finetuned_tokenizer = loadQlora(MODELO, output_dir)

# Now, use the loaded peft_finetuned_model and peft_finetuned_tokenizer with the generate_text function

prompt1 = "Hello, how are you today?"
response1 = generate_text(prompt1, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"Prompt: {prompt1}")
print(f"Response: {response1}")

prompt2 = "What is your favorite color?"
response2 = generate_text(prompt2, peft_finetuned_model, peft_finetuned_tokenizer)
print(f"\nPrompt: {prompt2}")
print(f"Response: {response2}")

print("Model interaction section updated to use loadQlora function.")